# Day 1: Your First AI Program

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

**Today:** call a model, read the reply object, check its cost, and see why the same question can give two different answers.

Every model used in this course ends in `:free`.

## Setup

1. `uv sync` from the project folder.
2. Add a `.env` file next to this notebook:

```
OPENROUTER_API_KEY=sk-or-...your key...
```

Free key: [openrouter.ai/keys](https://openrouter.ai/keys)

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# import os
# os.getenv('OPENROUTER_API_KEY')

`load_dotenv()` loads `.env` into the environment. `True` means it was found; `False` means it's missing, misnamed, or in the wrong folder.

The key is never typed into a cell or printed.

## 1. Connect to a model with ChatOpenRouter

In [4]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free")

In [5]:
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, client=<openrouter.sdk.OpenRouter object at 0x0000020DC0F891C0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='poolside/laguna-s-2.1:free', model_kwargs={})

- `ChatOpenRouter` is a **connection**, not the model itself. Nothing is sent yet.
- The model string is `maker/model:free`.

## 2. Send your first message with invoke

In [7]:
reply = model.invoke("What is LangChain, in two sentences?")

reply.text

'LangChain is an open-source framework that helps developers build applications powered by Large Language Models (LLMs) like ChatGPT. It provides tools and abstractions for connecting to LLMs, managing context, and integrating with external data sources.'

`invoke` sends the text and returns the reply. That's your first AI program.

## 3. The reply is an object, not a string

In [8]:
reply

AIMessage(content='LangChain is an open-source framework that helps developers build applications powered by Large Language Models (LLMs) like ChatGPT. It provides tools and abstractions for connecting to LLMs, managing context, and integrating with external data sources.', additional_kwargs={'reasoning_content': 'Hmm, the user is asking for a two-sentence explanation of LangChain. This is a straightforward request for a concise definition. \n\nLangChain is a framework for building LLM applications, so I should focus on its core purpose and key functionality in the first sentence. The second sentence should mention its practical applications and how it simplifies development. \n\nKeeping it simple and avoiding technical jargon will make it easy to understand.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Hmm, the user is asking for a two-sentence explanation of LangChain. This is a straightforward request for a concise definition. \n\nLang

An `AIMessage`, covered in Day 2. It has two ways to read the answer:

In [9]:
reply.text

'LangChain is an open-source framework that helps developers build applications powered by Large Language Models (LLMs) like ChatGPT. It provides tools and abstractions for connecting to LLMs, managing context, and integrating with external data sources.'

In [10]:
reply.content

'LangChain is an open-source framework that helps developers build applications powered by Large Language Models (LLMs) like ChatGPT. It provides tools and abstractions for connecting to LLMs, managing context, and integrating with external data sources.'

Same for plain text. They diverge once a reply has images or tool calls, hence `.text`, which always gives just the words.

## 4. Count the tokens with usage_metadata

In [11]:
reply.usage_metadata

{'input_tokens': 29,
 'output_tokens': 136,
 'total_tokens': 165,
 'input_token_details': {'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'reasoning': 114}}

- `input_tokens` sent, `output_tokens` returned. A **token** ≈ ¾ of a word.
- Look inside `output_token_details`:

In [12]:
reply.usage_metadata["output_token_details"]

{'reasoning': 114}

### Reasoning tokens: paid for, never shown

`reasoning` tokens are spent **thinking**, before the answer. Not in `reply.text`, not readable, and on a paid model, billed anyway. Most current models do this.

## 5. Read response_metadata: model, finish reason, cost

In [13]:
reply.response_metadata

{'model_name': 'poolside/laguna-s-2.1:free',
 'id': 'gen-1786188865-19L46uTiglnHyoldaUWL',
 'created': 1786188865,
 'object': 'chat.completion',
 'finish_reason': 'stop',
 'logprobs': None,
 'model_provider': 'openrouter',
 'cost': 0.0,
 'cost_details': {'upstream_inference_completions_cost': 0.0,
  'upstream_inference_prompt_cost': 0.0,
  'upstream_inference_cost': 0.0}}

- `model_name`: which model actually answered. Free models can be swapped or retired without notice.
- `finish_reason`: `stop` means it finished naturally (more in section 8).
- `cost`: `0.0`, proof the `:free` suffix worked.

## 6. Why the same question gives a different answer

Run the next cell two or three times.

In [16]:
model.invoke("Give me one interesting fact about the moon.").text

"One fascinating fact about the moon is that it is **moving away from Earth** at a rate of about **3.8 centimeters per year**. This happens because the Moon's gravitational pull pulls on Earth's oceans, causing the tidal bulge to move forward, which gradually increases the distance between the two."

In [ ]:
# "The moon is slowly drifting away from Earth — at a rate of about **3.8 centimeters per year**. Over millions of years, this gradual retreat has caused the length of a day on Earth to be longer (currently about 24 hours), and it's one of the reasons Earth's tides are gradually shifting. In about 1.5 billion years, the moon will be far enough away that Earth's rotation will no longer be synchronized with the moon's orbit, making a day significantly longer than 24 hours."

The answers differ. A model predicts the next word with deliberate randomness, it doesn't look one up.

**Same question, different answer** is normal, not a bug.

## 7. Control randomness with temperature

`temperature` controls how adventurous the next word choice is. `0` is as repeatable as it gets.

In [19]:
steady = ChatOpenRouter(model="nvidia/nemotron-3.5-lightning:free", temperature=0)

steady.invoke("Name the largest planet in our solar system.").text

'The largest planet in our solar system is **Jupiter**.'

Run twice: the answer should be the same or very close. Use `temperature=0` for facts and tests, higher for ideas and variety.

Even at zero, runs aren't guaranteed identical. Treat it as "as steady as it gets," not a promise.

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openrouter': '0.2.7'}}, client=<openrouter.sdk.OpenRouter object at 0x0000020DC22DA4B0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='poolside/laguna-s-2.1:free', temperature=0.0, model_kwargs={})

## 8. Capping the reply length with max_tokens

`max_tokens` caps the reply length. Set it absurdly low and watch what happens.

In [22]:
clipped = ChatOpenRouter(
    model="nvidia/nemotron-3.5-lightning:free",
    temperature=0,
    max_tokens=20,
)

answer = clipped.invoke("Explain how a jet engine works.")

answer.text

''

In [23]:
answer

AIMessage(content='', additional_kwargs={'reasoning_content': 'Hmm, the user is asking about how a jet engine works. This is a fairly common but detailed', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'Hmm, the user is asking about how a jet engine works. This is a fairly common but detailed'}]}, response_metadata={'model_name': 'poolside/laguna-s-2.1:free', 'id': 'gen-1786189438-MWPt9Vs6oPI4OTRj44CF', 'created': 1786189438, 'object': 'chat.completion', 'finish_reason': 'length', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0, 'cost_details': {'upstream_inference_completions_cost': 0.0, 'upstream_inference_prompt_cost': 0.0, 'upstream_inference_cost': 0.0}}, id='lc_run--019fe12f-d046-7593-9bef-fbc504d214bf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 27, 'output_tokens': 20, 'total_tokens': 47, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 23}}

Empty, not a half-finished sentence. Nothing at all.

In [24]:
answer.response_metadata["finish_reason"]

'length'

In [28]:
answer.usage_metadata

{'input_tokens': 27,
 'output_tokens': 20,
 'total_tokens': 47,
 'input_token_details': {'cache_read': 0, 'cache_creation': 0},
 'output_token_details': {'reasoning': 23}}

`finish_reason` is `length`, it hit the ceiling. The token budget went entirely to **reasoning**, the model never got to writing.

You paid in full for an empty string. Check `finish_reason`: an empty `reply.text` isn't always an error or a refusal, sometimes it's just too low a ceiling.